In [24]:
import copy
import calendar
from datetime import datetime, timedelta

import pandas as pd
import numpy as np
import cvxpy as cp
import matplotlib.pyplot as plt

from time_process import (
    generate_hourly_datetime_pairs, 
    get_month_range, 
    generate_day_pairs,
)

plt.rcParams['font.sans-serif']=['SimHei']    # 用来正常显示中文标签
plt.rcParams['axes.unicode_minus'] = False    # 用来显示负号

In [25]:
exp_name = "estimate1016"
node_name = "route_B"

定量   
d: 需量功率  
e_c：储能充放功率（正为放电，负为充电)  
p：电价  
e_r_1：目前储能器001的剩余电量  
e_r_2：目前储能器002的剩余电量  
e_l：每个时段损耗电量  
c_l: 充放电功率损失  
e_c_max：充放电功率上限  
e_c_min：充放电功率下限  
e_s_max: 储能器soc上限  
e_s_min：储能器soc下限  
lamda_v: 谷电时段矫正系数  
lamda_p: 峰电时段矫正系数  
lamda_f: 平电时段矫正系数  

变量   
e_c_1 储能系统001对外的充放电功率 kW  
e_c_2 储能系统002对外的充放电功率 kW  
soc_1 储能系统001电量 kWh
soc_2 储能系统002电量 kWh

In [26]:
class EsArbitraryRangeScheduler_withMaxDemand:
    # TODO 拆分设备参数和运行数据
    def __init__(self, 
                 schedule_time_range: list, 
                 demand_load, 
                 ele_prices, 
                 ele_types, 
                 devices_info, 
                 current_soc_list, 
                 max_demand_line,
                 is_slow_charge: bool = False):
        self.schedule_time_range = schedule_time_range
        self.schedule_time_length = len(self.schedule_time_range)
        self.demand_load = demand_load
        self.ele_prices = ele_prices
        self.ele_types = ele_types
        self.devices_num = len(devices_info)
        self.is_slow_charge = is_slow_charge
        self.current_soc_list = current_soc_list
        self.charge_loss_list = [i["charge_loss"] for i in devices_info]
        self.discharge_loss_list = [i["discharge_loss"] for i in devices_info]
        self.es_charge_max_list = [i["es_charge_max"] for i in devices_info]
        self.es_discharge_max_list = [i["es_charge_min"] for i in devices_info]
        self.es_capacity_max_list = [i["es_capacity_max"] * i["usable_depth"] for i in devices_info]
        self.es_capacity_min_list = [i["es_capacity_min"] for i in devices_info]
        self.max_demand_line = max(max_demand_line, max(demand_load))
    
    def modeling2solving(self):
        row = self.devices_num
        column = self.schedule_time_length

        #设备参数
        c_l_in_vec = np.array(self.charge_loss_list).reshape((row, 1))
        c_l_out_vec = np.array(self.discharge_loss_list).reshape((row, 1))
        e_c_max_vec = np.array(self.es_charge_max_list).reshape((row, 1))
        e_c_min_vec = np.array(self.es_discharge_max_list).reshape((row, 1))
        e_s_max_vec = np.array(self.es_capacity_max_list).reshape((row, 1))
        e_s_min_vec = np.array(self.es_capacity_min_list).reshape((row, 1))
        
        #充放电模式参数
        lamda_v = 0.0001
        lamda_f = 0.0001
        lamda_p = -3 * lamda_v
        lamda_tp = 2 * lamda_p

        lamda_amortize = 0.001

        time_ratio = 15/60

        # 输入定量
        d = np.array(self.demand_load)
        p = np.array(self.ele_prices)
        # e_r_vec = np.array([self.current_soc_list[i] / 100 * e_s_max_vec[i] for i in range(row)])
        # e_r_vec = np.array([self.current_soc_list[i] * e_s_max_vec[i] for i in range(row)])
        e_r_vec = np.array(self.current_soc_list)

        # 定义设备级变量
        e_c_in_matrix = cp.Variable((row, column))
        e_c_out_matrix = cp.Variable((row, column))
        soc_matrix = cp.Variable((row, column))
        # 定义节点级变量
        e_c_in_agg_vec = cp.sum(e_c_in_matrix, axis=0)
        e_c_out_agg_vec = cp.sum(e_c_out_matrix, axis=0)
        soc_agg_vec = cp.sum(soc_matrix, axis=0)

        # 目标函数
        profit = time_ratio * (e_c_in_agg_vec + e_c_out_agg_vec) @ p

        if self.is_slow_charge:
            profit = profit - lamda_amortize * cp.norm(e_c_in_agg_vec)

            for j in range(column):
                if self.ele_types[j] == "峰":
                    profit = profit + lamda_p * soc_agg_vec[j]
                elif self.ele_types[j] == "尖峰":
                    profit = profit + lamda_tp * soc_agg_vec[j]
        else:
            for j in range(column):
                if self.ele_types[j] == "谷":
                    profit = profit + lamda_v * soc_agg_vec[j]
                elif self.ele_types[j] == "峰":
                    profit = profit + lamda_p * soc_agg_vec[j]
                elif self.ele_types[j] == "尖峰":
                    profit = profit + lamda_tp * soc_agg_vec[j]
                elif self.ele_types[j] == "平":
                    profit = profit + lamda_f * soc_agg_vec[j]


        obj = cp.Maximize(profit)

        # 设置约束条件
        constraints = []

        # 充电功率和实时电量匹配
        for i in range(row):
            for j in range(column):
                constraints += [soc_matrix[i, j] == e_r_vec[i] \
                                - cp.sum(e_c_in_matrix[i, :j+1]) * time_ratio * c_l_in_vec[i] \
                                - cp.sum(e_c_out_matrix[i, :j+1]) * time_ratio / c_l_out_vec[i]]

        # 放电功率小于需量
        constraints += [e_c_out_agg_vec <= cp.pos(d)]

        # 总功率小于最大需量控制线
        constraints += [d - e_c_in_agg_vec <= self.max_demand_line]

        # 储能系统每个时段的充放电功率限制
        constraints += [e_c_out_matrix <= e_c_max_vec]
        constraints += [e_c_out_matrix >= 0]
        constraints += [e_c_in_matrix <= 0]
        constraints += [e_c_in_matrix >= e_c_min_vec]

        # 对电量损耗的保底电量限制
        # （此条限制在滚动策略中无法保证满足，建议在EMS中进行设置）
        # constraints += [soc >= e_s_max * 0.01]

        # 储能器容量限制
        constraints += [soc_matrix >= e_s_min_vec]
        constraints += [soc_matrix <= e_s_max_vec]

        # 峰谷平时段充放电矫正
        for j in range(column):
            if self.ele_types[j] == "谷":
                constraints += [e_c_out_agg_vec[j] == 0]
            elif self.ele_types[j] == "峰":
                constraints += [e_c_in_agg_vec[j] == 0]
            elif self.ele_types[j] == "尖峰":
                constraints += [e_c_in_agg_vec[j] == 0]
            elif self.ele_types[j] == "平":
                constraints += [e_c_out_agg_vec[j] == 0]

        prob = cp.Problem(obj, constraints)
        result = prob.solve(verbose = False, solver = cp.CLARABEL)
        return result, e_c_in_matrix.value, e_c_out_matrix.value
    
    def schedule_generate(self, charge_array, discharge_array):
        schedule_list = []
        
        for device_i in range(self.devices_num):
            power_array_i = charge_array[device_i] + discharge_array[device_i]
            power_array_i = np.around(power_array_i, decimals=3)
            
            for j in range(len(power_array_i)):
                if abs(power_array_i[j]) < 0.1:
                    power_array_i[j] = 0
            
            schedule_i_df = pd.DataFrame({"power_opt": power_array_i}, index=self.schedule_time_range)
            schedule_list.append(schedule_i_df)
        return schedule_list
    
    def run(self):
        profit, charge_array, discharge_array = self.modeling2solving()
        schedule_list = self.schedule_generate(charge_array, discharge_array)
        
        return schedule_list

In [27]:
devices_info = [{
    "usable_depth": 0.95,
    "charge_loss": 0.92,
    "discharge_loss": 0.95,
    "es_charge_max": 12500,
    "es_charge_min": -12500,
    "es_capacity_max": 25000,
    "es_capacity_min": 0,
}]

In [28]:
# def flat_valley_price_diff(ele_price_df):
#     flat_ele_price_df = copy.deepcopy(ele_price_df)
#     if flat_ele_price_df["type"].isin(["谷"]).any():
#         v_index = flat_ele_price_df[flat_ele_price_df["type"] == "谷"].index[-1]
#     else:
#         v_index = -1
#     if flat_ele_price_df["type"].isin(["深谷"]).any():
#         dv_index = flat_ele_price_df[flat_ele_price_df["type"] == "深谷"].index[-1]
#     else:
#         dv_index = -1
    
#     flat_price_index = max(v_index, dv_index)
#     flat_price = flat_ele_price_df.loc[flat_price_index, "value"]
    
#     flat_ele_price_df.loc[flat_ele_price_df['type'] == '谷', 'value'] = flat_price
#     flat_ele_price_df.loc[flat_ele_price_df['type'] == '深谷', 'value'] = flat_price
    
#     return flat_ele_price_df

In [29]:
def get_max_value_by_month(df: pd.DataFrame, target_time: datetime) -> float:
    # 提取目标月份和年份
    target_year = target_time.year
    target_month = target_time.month

    # 筛选出 time 列中年份和月份匹配的行
    mask = (df['time'].dt.year == target_year) & (df['time'].dt.month == target_month)
    filtered_df = df[mask]

    # 如果没有匹配的数据，返回 None
    if filtered_df.empty:
        return None
    # 返回 value 列的最大值
    return filtered_df['value'].max()


def get_max_value_by_day(df: pd.DataFrame, target_time: datetime) -> float:
    # 筛选出 time 列中年份和月份匹配的行
    if target_time == df["time"].min():
        filtered_df = df.loc[(df["time"] >= target_time) & (df["time"] < target_time + timedelta(days=1))]
    else:
        filtered_df = df.loc[(df["time"] >= target_time - timedelta(days=1)) & (df["time"] < target_time)]
    print(filtered_df)
    # 如果没有匹配的数据，返回 None
    if filtered_df.empty:
        return None
    # 返回 value 列的最大值
    return filtered_df['value'].max()

In [30]:
demand_load_df = pd.read_csv(f"./data/{exp_name}/{node_name}/demand_load.csv")
demand_load_df['time'] = pd.to_datetime(demand_load_df['time'])
demand_load_df

,time,value
0,2024-03-01 00:00:00,41800.0
1,2024-03-01 00:15:00,42075.0
2,2024-03-01 00:30:00,41250.0
3,2024-03-01 00:45:00,41800.0
4,2024-03-01 01:00:00,42075.0
...,...,...
35035,2025-02-28 22:45:00,58575.0
35036,2025-02-28 23:00:00,58575.0
35037,2025-02-28 23:15:00,57475.0
35038,2025-02-28 23:30:00,57475.0


In [31]:
ele_price_df = pd.read_csv(f"./data/{exp_name}/{node_name}/ele_price.csv")
ele_price_df['time'] = pd.to_datetime(ele_price_df['time'])
ele_price_df

,time,value,type
0,2024-03-01 00:00:00,0.293321,谷
1,2024-03-01 00:15:00,0.293321,谷
2,2024-03-01 00:30:00,0.293321,谷
3,2024-03-01 00:45:00,0.293321,谷
4,2024-03-01 01:00:00,0.293321,谷
...,...,...,...
35035,2025-02-28 22:45:00,0.559769,平
35036,2025-02-28 23:00:00,0.559769,平
35037,2025-02-28 23:15:00,0.559769,平
35038,2025-02-28 23:30:00,0.559769,平


In [32]:
save_range_start = datetime(2024, 3, 1, 0, 0, 0)
save_range_end = datetime(2025, 3, 1, 0, 0, 0)
validation_day_list = generate_day_pairs(save_range_start, save_range_end)
len(validation_day_list)
print(validation_day_list[0])

(datetime.datetime(2024, 3, 1, 0, 0), datetime.datetime(2024, 3, 2, 0, 0))


## P=0.0

In [33]:
days_strategy_list = []
for time_pair in validation_day_list:
    vs_time, ve_time = time_pair[0], time_pair[1]
    print(f"vs_time-ve_time: {vs_time}-{ve_time}")
    # demand load
    mask = (demand_load_df['time'] >= vs_time) & (demand_load_df['time'] < ve_time)
    step_demand_load_df = demand_load_df.loc[mask]
    # ele price
    mask = (ele_price_df['time'] >= vs_time) & (ele_price_df['time'] < ve_time)
    step_ele_price_df = ele_price_df.loc[mask]
    # max demand line
    max_demand_line = get_max_value_by_month(demand_load_df, vs_time)
    max_demand_line = max_demand_line * 1.00
    # scheduler model
    scheduler_model = EsArbitraryRangeScheduler_withMaxDemand(
        step_demand_load_df["time"].to_list(), 
        step_demand_load_df["value"].to_list(), 
        step_ele_price_df["value"].to_list(), 
        step_ele_price_df["type"].to_list(),
        devices_info,
        [0],
        max_demand_line,
    )
    opt_list = scheduler_model.run()
    # results
    days_strategy_list.append(opt_list[0])

result_df = pd.concat(days_strategy_list)
result_df["time"] = result_df.index
mask = (result_df['time'] >= save_range_start) & (result_df['time'] < save_range_end)
save_result_df = result_df.loc[mask]
save_result_df.to_csv(f"/Users/wangzf/work/benefits_estimation/es_schedule_for_MaxDemand_FuDing/data/{exp_name}/{node_name}/opt_result/schedule_result_no_exceed_break0percent.csv", encoding="utf-8", index=False)

vs_time-ve_time: 2024-03-01 00:00:00-2024-03-02 00:00:00
vs_time-ve_time: 2024-03-02 00:00:00-2024-03-03 00:00:00
vs_time-ve_time: 2024-03-03 00:00:00-2024-03-04 00:00:00
vs_time-ve_time: 2024-03-04 00:00:00-2024-03-05 00:00:00
vs_time-ve_time: 2024-03-05 00:00:00-2024-03-06 00:00:00
vs_time-ve_time: 2024-03-06 00:00:00-2024-03-07 00:00:00
vs_time-ve_time: 2024-03-07 00:00:00-2024-03-08 00:00:00
vs_time-ve_time: 2024-03-08 00:00:00-2024-03-09 00:00:00
vs_time-ve_time: 2024-03-09 00:00:00-2024-03-10 00:00:00
vs_time-ve_time: 2024-03-10 00:00:00-2024-03-11 00:00:00
vs_time-ve_time: 2024-03-11 00:00:00-2024-03-12 00:00:00
vs_time-ve_time: 2024-03-12 00:00:00-2024-03-13 00:00:00
vs_time-ve_time: 2024-03-13 00:00:00-2024-03-14 00:00:00
vs_time-ve_time: 2024-03-14 00:00:00-2024-03-15 00:00:00
vs_time-ve_time: 2024-03-15 00:00:00-2024-03-16 00:00:00
vs_time-ve_time: 2024-03-16 00:00:00-2024-03-17 00:00:00
vs_time-ve_time: 2024-03-17 00:00:00-2024-03-18 00:00:00
vs_time-ve_time: 2024-03-18 00:

## P=1

In [34]:
days_strategy_list = []
for time_pair in validation_day_list:
    vs_time, ve_time = time_pair[0], time_pair[1]
    print(f"vs_time-ve_time: {vs_time}-{ve_time}")
    # demand load
    mask = (demand_load_df['time'] >= vs_time) & (demand_load_df['time'] < ve_time)
    step_demand_load_df = demand_load_df.loc[mask]
    # ele price
    mask = (ele_price_df['time'] >= vs_time) & (ele_price_df['time'] < ve_time)
    step_ele_price_df = ele_price_df.loc[mask]
    # max demand line
    max_demand_line = get_max_value_by_month(demand_load_df, vs_time)
    max_demand_line = max_demand_line * 1.01
    # scheduler model
    scheduler_model = EsArbitraryRangeScheduler_withMaxDemand(
        step_demand_load_df["time"].to_list(), 
        step_demand_load_df["value"].to_list(), 
        step_ele_price_df["value"].to_list(), 
        step_ele_price_df["type"].to_list(),
        devices_info,
        [0],
        max_demand_line,
    )
    opt_list = scheduler_model.run()
    # results
    days_strategy_list.append(opt_list[0])

result_df = pd.concat(days_strategy_list)
result_df["time"] = result_df.index
mask = (result_df['time'] >= save_range_start) & (result_df['time'] < save_range_end)
save_result_df = result_df.loc[mask]
save_result_df.to_csv(f"/Users/wangzf/work/benefits_estimation/es_schedule_for_MaxDemand_FuDing/data/{exp_name}/{node_name}/opt_result/schedule_result_no_exceed_break1percent.csv", encoding="utf-8", index=False)

vs_time-ve_time: 2024-03-01 00:00:00-2024-03-02 00:00:00
vs_time-ve_time: 2024-03-02 00:00:00-2024-03-03 00:00:00
vs_time-ve_time: 2024-03-03 00:00:00-2024-03-04 00:00:00
vs_time-ve_time: 2024-03-04 00:00:00-2024-03-05 00:00:00
vs_time-ve_time: 2024-03-05 00:00:00-2024-03-06 00:00:00
vs_time-ve_time: 2024-03-06 00:00:00-2024-03-07 00:00:00
vs_time-ve_time: 2024-03-07 00:00:00-2024-03-08 00:00:00
vs_time-ve_time: 2024-03-08 00:00:00-2024-03-09 00:00:00
vs_time-ve_time: 2024-03-09 00:00:00-2024-03-10 00:00:00
vs_time-ve_time: 2024-03-10 00:00:00-2024-03-11 00:00:00
vs_time-ve_time: 2024-03-11 00:00:00-2024-03-12 00:00:00
vs_time-ve_time: 2024-03-12 00:00:00-2024-03-13 00:00:00
vs_time-ve_time: 2024-03-13 00:00:00-2024-03-14 00:00:00
vs_time-ve_time: 2024-03-14 00:00:00-2024-03-15 00:00:00
vs_time-ve_time: 2024-03-15 00:00:00-2024-03-16 00:00:00
vs_time-ve_time: 2024-03-16 00:00:00-2024-03-17 00:00:00
vs_time-ve_time: 2024-03-17 00:00:00-2024-03-18 00:00:00
vs_time-ve_time: 2024-03-18 00:

## P=2

In [35]:
days_strategy_list = []
for time_pair in validation_day_list:
    vs_time, ve_time = time_pair[0], time_pair[1]
    print(f"vs_time-ve_time: {vs_time}-{ve_time}")
    # demand load
    mask = (demand_load_df['time'] >= vs_time) & (demand_load_df['time'] < ve_time)
    step_demand_load_df = demand_load_df.loc[mask]
    # ele price
    mask = (ele_price_df['time'] >= vs_time) & (ele_price_df['time'] < ve_time)
    step_ele_price_df = ele_price_df.loc[mask]
    # max demand line
    max_demand_line = get_max_value_by_month(demand_load_df, vs_time)
    max_demand_line = max_demand_line * 1.02
    # scheduler model
    scheduler_model = EsArbitraryRangeScheduler_withMaxDemand(
        step_demand_load_df["time"].to_list(), 
        step_demand_load_df["value"].to_list(), 
        step_ele_price_df["value"].to_list(), 
        step_ele_price_df["type"].to_list(),
        devices_info,
        [0],
        max_demand_line,
    )
    opt_list = scheduler_model.run()
    # results
    days_strategy_list.append(opt_list[0])

result_df = pd.concat(days_strategy_list)
result_df["time"] = result_df.index
mask = (result_df['time'] >= save_range_start) & (result_df['time'] < save_range_end)
save_result_df = result_df.loc[mask]
save_result_df.to_csv(f"/Users/wangzf/work/benefits_estimation/es_schedule_for_MaxDemand_FuDing/data/{exp_name}/{node_name}/opt_result/schedule_result_no_exceed_break2percent.csv", encoding="utf-8", index=False)

vs_time-ve_time: 2024-03-01 00:00:00-2024-03-02 00:00:00
vs_time-ve_time: 2024-03-02 00:00:00-2024-03-03 00:00:00
vs_time-ve_time: 2024-03-03 00:00:00-2024-03-04 00:00:00
vs_time-ve_time: 2024-03-04 00:00:00-2024-03-05 00:00:00
vs_time-ve_time: 2024-03-05 00:00:00-2024-03-06 00:00:00
vs_time-ve_time: 2024-03-06 00:00:00-2024-03-07 00:00:00
vs_time-ve_time: 2024-03-07 00:00:00-2024-03-08 00:00:00
vs_time-ve_time: 2024-03-08 00:00:00-2024-03-09 00:00:00
vs_time-ve_time: 2024-03-09 00:00:00-2024-03-10 00:00:00
vs_time-ve_time: 2024-03-10 00:00:00-2024-03-11 00:00:00
vs_time-ve_time: 2024-03-11 00:00:00-2024-03-12 00:00:00
vs_time-ve_time: 2024-03-12 00:00:00-2024-03-13 00:00:00
vs_time-ve_time: 2024-03-13 00:00:00-2024-03-14 00:00:00
vs_time-ve_time: 2024-03-14 00:00:00-2024-03-15 00:00:00
vs_time-ve_time: 2024-03-15 00:00:00-2024-03-16 00:00:00
vs_time-ve_time: 2024-03-16 00:00:00-2024-03-17 00:00:00
vs_time-ve_time: 2024-03-17 00:00:00-2024-03-18 00:00:00
vs_time-ve_time: 2024-03-18 00:

## P=3

In [36]:
days_strategy_list = []
for time_pair in validation_day_list:
    vs_time, ve_time = time_pair[0], time_pair[1]
    print(f"vs_time-ve_time: {vs_time}-{ve_time}")
    # demand load
    mask = (demand_load_df['time'] >= vs_time) & (demand_load_df['time'] < ve_time)
    step_demand_load_df = demand_load_df.loc[mask]
    # ele price
    mask = (ele_price_df['time'] >= vs_time) & (ele_price_df['time'] < ve_time)
    step_ele_price_df = ele_price_df.loc[mask]
    # max demand line
    max_demand_line = get_max_value_by_month(demand_load_df, vs_time)
    max_demand_line = max_demand_line * 1.03
    # scheduler model
    scheduler_model = EsArbitraryRangeScheduler_withMaxDemand(
        step_demand_load_df["time"].to_list(), 
        step_demand_load_df["value"].to_list(), 
        step_ele_price_df["value"].to_list(), 
        step_ele_price_df["type"].to_list(),
        devices_info,
        [0],
        max_demand_line,
    )
    opt_list = scheduler_model.run()
    # results
    days_strategy_list.append(opt_list[0])

result_df = pd.concat(days_strategy_list)
result_df["time"] = result_df.index
mask = (result_df['time'] >= save_range_start) & (result_df['time'] < save_range_end)
save_result_df = result_df.loc[mask]
save_result_df.to_csv(f"/Users/wangzf/work/benefits_estimation/es_schedule_for_MaxDemand_FuDing/data/{exp_name}/{node_name}/opt_result/schedule_result_no_exceed_break3percent.csv", encoding="utf-8", index=False)

vs_time-ve_time: 2024-03-01 00:00:00-2024-03-02 00:00:00
vs_time-ve_time: 2024-03-02 00:00:00-2024-03-03 00:00:00
vs_time-ve_time: 2024-03-03 00:00:00-2024-03-04 00:00:00
vs_time-ve_time: 2024-03-04 00:00:00-2024-03-05 00:00:00
vs_time-ve_time: 2024-03-05 00:00:00-2024-03-06 00:00:00
vs_time-ve_time: 2024-03-06 00:00:00-2024-03-07 00:00:00
vs_time-ve_time: 2024-03-07 00:00:00-2024-03-08 00:00:00
vs_time-ve_time: 2024-03-08 00:00:00-2024-03-09 00:00:00
vs_time-ve_time: 2024-03-09 00:00:00-2024-03-10 00:00:00
vs_time-ve_time: 2024-03-10 00:00:00-2024-03-11 00:00:00
vs_time-ve_time: 2024-03-11 00:00:00-2024-03-12 00:00:00
vs_time-ve_time: 2024-03-12 00:00:00-2024-03-13 00:00:00
vs_time-ve_time: 2024-03-13 00:00:00-2024-03-14 00:00:00
vs_time-ve_time: 2024-03-14 00:00:00-2024-03-15 00:00:00
vs_time-ve_time: 2024-03-15 00:00:00-2024-03-16 00:00:00
vs_time-ve_time: 2024-03-16 00:00:00-2024-03-17 00:00:00
vs_time-ve_time: 2024-03-17 00:00:00-2024-03-18 00:00:00
vs_time-ve_time: 2024-03-18 00:

## P=4

In [37]:
days_strategy_list = []
for time_pair in validation_day_list:
    vs_time, ve_time = time_pair[0], time_pair[1]
    print(f"vs_time-ve_time: {vs_time}-{ve_time}")
    # demand load
    mask = (demand_load_df['time'] >= vs_time) & (demand_load_df['time'] < ve_time)
    step_demand_load_df = demand_load_df.loc[mask]
    # ele price
    mask = (ele_price_df['time'] >= vs_time) & (ele_price_df['time'] < ve_time)
    step_ele_price_df = ele_price_df.loc[mask]
    # max demand line
    max_demand_line = get_max_value_by_month(demand_load_df, vs_time)
    max_demand_line = max_demand_line * 1.04
    # scheduler model
    scheduler_model = EsArbitraryRangeScheduler_withMaxDemand(
        step_demand_load_df["time"].to_list(), 
        step_demand_load_df["value"].to_list(), 
        step_ele_price_df["value"].to_list(), 
        step_ele_price_df["type"].to_list(),
        devices_info,
        [0],
        max_demand_line,
    )
    opt_list = scheduler_model.run()
    # results
    days_strategy_list.append(opt_list[0])

result_df = pd.concat(days_strategy_list)
result_df["time"] = result_df.index
mask = (result_df['time'] >= save_range_start) & (result_df['time'] < save_range_end)
save_result_df = result_df.loc[mask]
save_result_df.to_csv(f"/Users/wangzf/work/benefits_estimation/es_schedule_for_MaxDemand_FuDing/data/{exp_name}/{node_name}/opt_result/schedule_result_no_exceed_break4percent.csv", encoding="utf-8", index=False)

vs_time-ve_time: 2024-03-01 00:00:00-2024-03-02 00:00:00
vs_time-ve_time: 2024-03-02 00:00:00-2024-03-03 00:00:00
vs_time-ve_time: 2024-03-03 00:00:00-2024-03-04 00:00:00
vs_time-ve_time: 2024-03-04 00:00:00-2024-03-05 00:00:00
vs_time-ve_time: 2024-03-05 00:00:00-2024-03-06 00:00:00
vs_time-ve_time: 2024-03-06 00:00:00-2024-03-07 00:00:00
vs_time-ve_time: 2024-03-07 00:00:00-2024-03-08 00:00:00
vs_time-ve_time: 2024-03-08 00:00:00-2024-03-09 00:00:00
vs_time-ve_time: 2024-03-09 00:00:00-2024-03-10 00:00:00
vs_time-ve_time: 2024-03-10 00:00:00-2024-03-11 00:00:00
vs_time-ve_time: 2024-03-11 00:00:00-2024-03-12 00:00:00
vs_time-ve_time: 2024-03-12 00:00:00-2024-03-13 00:00:00
vs_time-ve_time: 2024-03-13 00:00:00-2024-03-14 00:00:00
vs_time-ve_time: 2024-03-14 00:00:00-2024-03-15 00:00:00
vs_time-ve_time: 2024-03-15 00:00:00-2024-03-16 00:00:00
vs_time-ve_time: 2024-03-16 00:00:00-2024-03-17 00:00:00
vs_time-ve_time: 2024-03-17 00:00:00-2024-03-18 00:00:00
vs_time-ve_time: 2024-03-18 00:

## P=5

In [38]:
days_strategy_list = []
for time_pair in validation_day_list:
    vs_time, ve_time = time_pair[0], time_pair[1]
    print(f"vs_time-ve_time: {vs_time}-{ve_time}")
    # demand load
    mask = (demand_load_df['time'] >= vs_time) & (demand_load_df['time'] < ve_time)
    step_demand_load_df = demand_load_df.loc[mask]
    # ele price
    mask = (ele_price_df['time'] >= vs_time) & (ele_price_df['time'] < ve_time)
    step_ele_price_df = ele_price_df.loc[mask]
    # max demand line
    max_demand_line = get_max_value_by_month(demand_load_df, vs_time)
    max_demand_line = max_demand_line * 1.05
    # scheduler model
    scheduler_model = EsArbitraryRangeScheduler_withMaxDemand(
        step_demand_load_df["time"].to_list(), 
        step_demand_load_df["value"].to_list(), 
        step_ele_price_df["value"].to_list(), 
        step_ele_price_df["type"].to_list(),
        devices_info,
        [0],
        max_demand_line,
    )
    opt_list = scheduler_model.run()
    # results
    days_strategy_list.append(opt_list[0])

result_df = pd.concat(days_strategy_list)
result_df["time"] = result_df.index
mask = (result_df['time'] >= save_range_start) & (result_df['time'] < save_range_end)
save_result_df = result_df.loc[mask]
save_result_df.to_csv(f"/Users/wangzf/work/benefits_estimation/es_schedule_for_MaxDemand_FuDing/data/{exp_name}/{node_name}/opt_result/schedule_result_no_exceed_break5percent.csv", encoding="utf-8", index=False)

vs_time-ve_time: 2024-03-01 00:00:00-2024-03-02 00:00:00
vs_time-ve_time: 2024-03-02 00:00:00-2024-03-03 00:00:00
vs_time-ve_time: 2024-03-03 00:00:00-2024-03-04 00:00:00
vs_time-ve_time: 2024-03-04 00:00:00-2024-03-05 00:00:00
vs_time-ve_time: 2024-03-05 00:00:00-2024-03-06 00:00:00
vs_time-ve_time: 2024-03-06 00:00:00-2024-03-07 00:00:00
vs_time-ve_time: 2024-03-07 00:00:00-2024-03-08 00:00:00
vs_time-ve_time: 2024-03-08 00:00:00-2024-03-09 00:00:00
vs_time-ve_time: 2024-03-09 00:00:00-2024-03-10 00:00:00
vs_time-ve_time: 2024-03-10 00:00:00-2024-03-11 00:00:00
vs_time-ve_time: 2024-03-11 00:00:00-2024-03-12 00:00:00
vs_time-ve_time: 2024-03-12 00:00:00-2024-03-13 00:00:00
vs_time-ve_time: 2024-03-13 00:00:00-2024-03-14 00:00:00
vs_time-ve_time: 2024-03-14 00:00:00-2024-03-15 00:00:00
vs_time-ve_time: 2024-03-15 00:00:00-2024-03-16 00:00:00
vs_time-ve_time: 2024-03-16 00:00:00-2024-03-17 00:00:00
vs_time-ve_time: 2024-03-17 00:00:00-2024-03-18 00:00:00
vs_time-ve_time: 2024-03-18 00:

## P=6

In [39]:
days_strategy_list = []
for time_pair in validation_day_list:
    vs_time, ve_time = time_pair[0], time_pair[1]
    print(f"vs_time-ve_time: {vs_time}-{ve_time}")
    # demand load
    mask = (demand_load_df['time'] >= vs_time) & (demand_load_df['time'] < ve_time)
    step_demand_load_df = demand_load_df.loc[mask]
    # ele price
    mask = (ele_price_df['time'] >= vs_time) & (ele_price_df['time'] < ve_time)
    step_ele_price_df = ele_price_df.loc[mask]
    # max demand line
    max_demand_line = get_max_value_by_month(demand_load_df, vs_time)
    max_demand_line = max_demand_line * 1.06
    # scheduler model
    scheduler_model = EsArbitraryRangeScheduler_withMaxDemand(
        step_demand_load_df["time"].to_list(), 
        step_demand_load_df["value"].to_list(), 
        step_ele_price_df["value"].to_list(), 
        step_ele_price_df["type"].to_list(),
        devices_info,
        [0],
        max_demand_line,
    )
    opt_list = scheduler_model.run()
    # results
    days_strategy_list.append(opt_list[0])

result_df = pd.concat(days_strategy_list)
result_df["time"] = result_df.index
mask = (result_df['time'] >= save_range_start) & (result_df['time'] < save_range_end)
save_result_df = result_df.loc[mask]
save_result_df.to_csv(f"/Users/wangzf/work/benefits_estimation/es_schedule_for_MaxDemand_FuDing/data/{exp_name}/{node_name}/opt_result/schedule_result_no_exceed_break6percent.csv", encoding="utf-8", index=False)

vs_time-ve_time: 2024-03-01 00:00:00-2024-03-02 00:00:00
vs_time-ve_time: 2024-03-02 00:00:00-2024-03-03 00:00:00
vs_time-ve_time: 2024-03-03 00:00:00-2024-03-04 00:00:00
vs_time-ve_time: 2024-03-04 00:00:00-2024-03-05 00:00:00
vs_time-ve_time: 2024-03-05 00:00:00-2024-03-06 00:00:00
vs_time-ve_time: 2024-03-06 00:00:00-2024-03-07 00:00:00
vs_time-ve_time: 2024-03-07 00:00:00-2024-03-08 00:00:00
vs_time-ve_time: 2024-03-08 00:00:00-2024-03-09 00:00:00
vs_time-ve_time: 2024-03-09 00:00:00-2024-03-10 00:00:00
vs_time-ve_time: 2024-03-10 00:00:00-2024-03-11 00:00:00
vs_time-ve_time: 2024-03-11 00:00:00-2024-03-12 00:00:00
vs_time-ve_time: 2024-03-12 00:00:00-2024-03-13 00:00:00
vs_time-ve_time: 2024-03-13 00:00:00-2024-03-14 00:00:00
vs_time-ve_time: 2024-03-14 00:00:00-2024-03-15 00:00:00
vs_time-ve_time: 2024-03-15 00:00:00-2024-03-16 00:00:00
vs_time-ve_time: 2024-03-16 00:00:00-2024-03-17 00:00:00
vs_time-ve_time: 2024-03-17 00:00:00-2024-03-18 00:00:00
vs_time-ve_time: 2024-03-18 00:

## P=7

In [40]:
days_strategy_list = []
for time_pair in validation_day_list:
    vs_time, ve_time = time_pair[0], time_pair[1]
    print(f"vs_time-ve_time: {vs_time}-{ve_time}")
    # demand load
    mask = (demand_load_df['time'] >= vs_time) & (demand_load_df['time'] < ve_time)
    step_demand_load_df = demand_load_df.loc[mask]
    # ele price
    mask = (ele_price_df['time'] >= vs_time) & (ele_price_df['time'] < ve_time)
    step_ele_price_df = ele_price_df.loc[mask]
    # max demand line
    max_demand_line = get_max_value_by_month(demand_load_df, vs_time)
    max_demand_line = max_demand_line * 1.07
    # scheduler model
    scheduler_model = EsArbitraryRangeScheduler_withMaxDemand(
        step_demand_load_df["time"].to_list(), 
        step_demand_load_df["value"].to_list(), 
        step_ele_price_df["value"].to_list(), 
        step_ele_price_df["type"].to_list(),
        devices_info,
        [0],
        max_demand_line,
    )
    opt_list = scheduler_model.run()
    # results
    days_strategy_list.append(opt_list[0])

result_df = pd.concat(days_strategy_list)
result_df["time"] = result_df.index
mask = (result_df['time'] >= save_range_start) & (result_df['time'] < save_range_end)
save_result_df = result_df.loc[mask]
save_result_df.to_csv(f"/Users/wangzf/work/benefits_estimation/es_schedule_for_MaxDemand_FuDing/data/{exp_name}/{node_name}/opt_result/schedule_result_no_exceed_break7percent.csv", encoding="utf-8", index=False)

vs_time-ve_time: 2024-03-01 00:00:00-2024-03-02 00:00:00
vs_time-ve_time: 2024-03-02 00:00:00-2024-03-03 00:00:00
vs_time-ve_time: 2024-03-03 00:00:00-2024-03-04 00:00:00
vs_time-ve_time: 2024-03-04 00:00:00-2024-03-05 00:00:00
vs_time-ve_time: 2024-03-05 00:00:00-2024-03-06 00:00:00
vs_time-ve_time: 2024-03-06 00:00:00-2024-03-07 00:00:00
vs_time-ve_time: 2024-03-07 00:00:00-2024-03-08 00:00:00
vs_time-ve_time: 2024-03-08 00:00:00-2024-03-09 00:00:00
vs_time-ve_time: 2024-03-09 00:00:00-2024-03-10 00:00:00
vs_time-ve_time: 2024-03-10 00:00:00-2024-03-11 00:00:00
vs_time-ve_time: 2024-03-11 00:00:00-2024-03-12 00:00:00
vs_time-ve_time: 2024-03-12 00:00:00-2024-03-13 00:00:00
vs_time-ve_time: 2024-03-13 00:00:00-2024-03-14 00:00:00
vs_time-ve_time: 2024-03-14 00:00:00-2024-03-15 00:00:00
vs_time-ve_time: 2024-03-15 00:00:00-2024-03-16 00:00:00
vs_time-ve_time: 2024-03-16 00:00:00-2024-03-17 00:00:00
vs_time-ve_time: 2024-03-17 00:00:00-2024-03-18 00:00:00
vs_time-ve_time: 2024-03-18 00:

## P=8

In [41]:
days_strategy_list = []
for time_pair in validation_day_list:
    vs_time, ve_time = time_pair[0], time_pair[1]
    print(f"vs_time-ve_time: {vs_time}-{ve_time}")
    # demand load
    mask = (demand_load_df['time'] >= vs_time) & (demand_load_df['time'] < ve_time)
    step_demand_load_df = demand_load_df.loc[mask]
    # ele price
    mask = (ele_price_df['time'] >= vs_time) & (ele_price_df['time'] < ve_time)
    step_ele_price_df = ele_price_df.loc[mask]
    # max demand line
    max_demand_line = get_max_value_by_month(demand_load_df, vs_time)
    max_demand_line = max_demand_line * 1.08
    # scheduler model
    scheduler_model = EsArbitraryRangeScheduler_withMaxDemand(
        step_demand_load_df["time"].to_list(), 
        step_demand_load_df["value"].to_list(), 
        step_ele_price_df["value"].to_list(), 
        step_ele_price_df["type"].to_list(),
        devices_info,
        [0],
        max_demand_line,
    )
    opt_list = scheduler_model.run()
    # results
    days_strategy_list.append(opt_list[0])

result_df = pd.concat(days_strategy_list)
result_df["time"] = result_df.index
mask = (result_df['time'] >= save_range_start) & (result_df['time'] < save_range_end)
save_result_df = result_df.loc[mask]
save_result_df.to_csv(f"/Users/wangzf/work/benefits_estimation/es_schedule_for_MaxDemand_FuDing/data/{exp_name}/{node_name}/opt_result/schedule_result_no_exceed_break8percent.csv", encoding="utf-8", index=False)

vs_time-ve_time: 2024-03-01 00:00:00-2024-03-02 00:00:00
vs_time-ve_time: 2024-03-02 00:00:00-2024-03-03 00:00:00
vs_time-ve_time: 2024-03-03 00:00:00-2024-03-04 00:00:00
vs_time-ve_time: 2024-03-04 00:00:00-2024-03-05 00:00:00
vs_time-ve_time: 2024-03-05 00:00:00-2024-03-06 00:00:00
vs_time-ve_time: 2024-03-06 00:00:00-2024-03-07 00:00:00
vs_time-ve_time: 2024-03-07 00:00:00-2024-03-08 00:00:00
vs_time-ve_time: 2024-03-08 00:00:00-2024-03-09 00:00:00
vs_time-ve_time: 2024-03-09 00:00:00-2024-03-10 00:00:00
vs_time-ve_time: 2024-03-10 00:00:00-2024-03-11 00:00:00
vs_time-ve_time: 2024-03-11 00:00:00-2024-03-12 00:00:00
vs_time-ve_time: 2024-03-12 00:00:00-2024-03-13 00:00:00
vs_time-ve_time: 2024-03-13 00:00:00-2024-03-14 00:00:00
vs_time-ve_time: 2024-03-14 00:00:00-2024-03-15 00:00:00
vs_time-ve_time: 2024-03-15 00:00:00-2024-03-16 00:00:00
vs_time-ve_time: 2024-03-16 00:00:00-2024-03-17 00:00:00
vs_time-ve_time: 2024-03-17 00:00:00-2024-03-18 00:00:00
vs_time-ve_time: 2024-03-18 00:

## P=9

In [42]:
days_strategy_list = []
for time_pair in validation_day_list:
    vs_time, ve_time = time_pair[0], time_pair[1]
    print(f"vs_time-ve_time: {vs_time}-{ve_time}")
    # demand load
    mask = (demand_load_df['time'] >= vs_time) & (demand_load_df['time'] < ve_time)
    step_demand_load_df = demand_load_df.loc[mask]
    # ele price
    mask = (ele_price_df['time'] >= vs_time) & (ele_price_df['time'] < ve_time)
    step_ele_price_df = ele_price_df.loc[mask]
    # max demand line
    max_demand_line = get_max_value_by_month(demand_load_df, vs_time)
    max_demand_line = max_demand_line * 1.09
    # scheduler model
    scheduler_model = EsArbitraryRangeScheduler_withMaxDemand(
        step_demand_load_df["time"].to_list(), 
        step_demand_load_df["value"].to_list(), 
        step_ele_price_df["value"].to_list(), 
        step_ele_price_df["type"].to_list(),
        devices_info,
        [0],
        max_demand_line,
    )
    opt_list = scheduler_model.run()
    # results
    days_strategy_list.append(opt_list[0])

result_df = pd.concat(days_strategy_list)
result_df["time"] = result_df.index
mask = (result_df['time'] >= save_range_start) & (result_df['time'] < save_range_end)
save_result_df = result_df.loc[mask]
save_result_df.to_csv(f"/Users/wangzf/work/benefits_estimation/es_schedule_for_MaxDemand_FuDing/data/{exp_name}/{node_name}/opt_result/schedule_result_no_exceed_break9percent.csv", encoding="utf-8", index=False)

vs_time-ve_time: 2024-03-01 00:00:00-2024-03-02 00:00:00
vs_time-ve_time: 2024-03-02 00:00:00-2024-03-03 00:00:00
vs_time-ve_time: 2024-03-03 00:00:00-2024-03-04 00:00:00
vs_time-ve_time: 2024-03-04 00:00:00-2024-03-05 00:00:00
vs_time-ve_time: 2024-03-05 00:00:00-2024-03-06 00:00:00
vs_time-ve_time: 2024-03-06 00:00:00-2024-03-07 00:00:00
vs_time-ve_time: 2024-03-07 00:00:00-2024-03-08 00:00:00
vs_time-ve_time: 2024-03-08 00:00:00-2024-03-09 00:00:00
vs_time-ve_time: 2024-03-09 00:00:00-2024-03-10 00:00:00
vs_time-ve_time: 2024-03-10 00:00:00-2024-03-11 00:00:00
vs_time-ve_time: 2024-03-11 00:00:00-2024-03-12 00:00:00
vs_time-ve_time: 2024-03-12 00:00:00-2024-03-13 00:00:00
vs_time-ve_time: 2024-03-13 00:00:00-2024-03-14 00:00:00
vs_time-ve_time: 2024-03-14 00:00:00-2024-03-15 00:00:00
vs_time-ve_time: 2024-03-15 00:00:00-2024-03-16 00:00:00
vs_time-ve_time: 2024-03-16 00:00:00-2024-03-17 00:00:00
vs_time-ve_time: 2024-03-17 00:00:00-2024-03-18 00:00:00
vs_time-ve_time: 2024-03-18 00:

## P=10

In [43]:
days_strategy_list = []
for time_pair in validation_day_list:
    vs_time, ve_time = time_pair[0], time_pair[1]
    print(f"vs_time-ve_time: {vs_time}-{ve_time}")
    # demand load
    mask = (demand_load_df['time'] >= vs_time) & (demand_load_df['time'] < ve_time)
    step_demand_load_df = demand_load_df.loc[mask]
    # ele price
    mask = (ele_price_df['time'] >= vs_time) & (ele_price_df['time'] < ve_time)
    step_ele_price_df = ele_price_df.loc[mask]
    # max demand line
    max_demand_line = get_max_value_by_month(demand_load_df, vs_time)
    max_demand_line = max_demand_line * 1.10
    # scheduler model
    scheduler_model = EsArbitraryRangeScheduler_withMaxDemand(
        step_demand_load_df["time"].to_list(), 
        step_demand_load_df["value"].to_list(), 
        step_ele_price_df["value"].to_list(), 
        step_ele_price_df["type"].to_list(),
        devices_info,
        [0],
        max_demand_line,
    )
    opt_list = scheduler_model.run()
    # results
    days_strategy_list.append(opt_list[0])

result_df = pd.concat(days_strategy_list)
result_df["time"] = result_df.index
mask = (result_df['time'] >= save_range_start) & (result_df['time'] < save_range_end)
save_result_df = result_df.loc[mask]
save_result_df.to_csv(f"/Users/wangzf/work/benefits_estimation/es_schedule_for_MaxDemand_FuDing/data/{exp_name}/{node_name}/opt_result/schedule_result_no_exceed_break10percent.csv", encoding="utf-8", index=False)

vs_time-ve_time: 2024-03-01 00:00:00-2024-03-02 00:00:00
vs_time-ve_time: 2024-03-02 00:00:00-2024-03-03 00:00:00
vs_time-ve_time: 2024-03-03 00:00:00-2024-03-04 00:00:00
vs_time-ve_time: 2024-03-04 00:00:00-2024-03-05 00:00:00
vs_time-ve_time: 2024-03-05 00:00:00-2024-03-06 00:00:00
vs_time-ve_time: 2024-03-06 00:00:00-2024-03-07 00:00:00
vs_time-ve_time: 2024-03-07 00:00:00-2024-03-08 00:00:00
vs_time-ve_time: 2024-03-08 00:00:00-2024-03-09 00:00:00
vs_time-ve_time: 2024-03-09 00:00:00-2024-03-10 00:00:00
vs_time-ve_time: 2024-03-10 00:00:00-2024-03-11 00:00:00
vs_time-ve_time: 2024-03-11 00:00:00-2024-03-12 00:00:00
vs_time-ve_time: 2024-03-12 00:00:00-2024-03-13 00:00:00
vs_time-ve_time: 2024-03-13 00:00:00-2024-03-14 00:00:00
vs_time-ve_time: 2024-03-14 00:00:00-2024-03-15 00:00:00
vs_time-ve_time: 2024-03-15 00:00:00-2024-03-16 00:00:00
vs_time-ve_time: 2024-03-16 00:00:00-2024-03-17 00:00:00
vs_time-ve_time: 2024-03-17 00:00:00-2024-03-18 00:00:00
vs_time-ve_time: 2024-03-18 00: